# 1. Environment Setup & Ollama GPU Initialization

In [1]:
import os
import subprocess
import platform
import time
import sys

# Start the Ollama server in the background
process = subprocess.Popen(
        ["ollama", "serve"], 
        stdout=subprocess.PIPE, 
        stderr=subprocess.PIPE,
        creationflags=subprocess.CREATE_NO_WINDOW if platform.system() == "Windows" else 0
)
time.sleep(5)
print("Ollama server started in the background.")

Ollama server started in the background.


In [2]:
import requests

ollama_url = "http://localhost:11434"
timeout_seconds = 60
start_time = time.time()

print(f"Waiting for Ollama server to be ready at {ollama_url}...")

while time.time() - start_time < timeout_seconds:
    try:
        response = requests.get(ollama_url)
        response.raise_for_status()
        print("✅ Ollama server is ready.")
        break
    except requests.exceptions.ConnectionError:
        print("Connection refused, retrying...")
        time.sleep(2)
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        time.sleep(2)
else:
    print("❌ Timeout waiting for Ollama server to be ready.")

Waiting for Ollama server to be ready at http://localhost:11434...
✅ Ollama server is ready.


In [3]:
import ollama

def pull_model(model_name):
    print(f"Attempting to pull model: {model_name}")
    try:
        for progress in ollama.pull(model_name, stream=False):
            print(f"Pulling {model_name}: {progress}", file=sys.stdout)
        print(f"✅ Successfully pulled model: {model_name}")
    except ollama.ResponseError as e:
        print(f"❌ Error pulling model {model_name}: {e}")
    except Exception as e:
        print(f"⚠️ Unexpected error: {e}")

models_to_pull = ["gemma3:4b"]
for model in models_to_pull:
    pull_model(model)

Attempting to pull model: gemma3:4b
Pulling gemma3:4b: ('status', 'success')
Pulling gemma3:4b: ('completed', None)
Pulling gemma3:4b: ('total', None)
Pulling gemma3:4b: ('digest', None)
✅ Successfully pulled model: gemma3:4b


# 2. LLM Query Helpers

In [4]:
def query_gpt_qwen(prompt):
    model_name = 'gemma3:4b'
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are an expert in image enhancement and Python coding."},
            {"role": "user", "content": prompt}
        ]
    )
    return response['message']['content']

In [5]:
def query_gpt_llava(image_input, previous_output=None):
    import numpy as np
    from PIL import Image
    import tempfile

    temp_file = None
    try:
        if isinstance(image_input, str):
            image_path = image_input
        elif isinstance(image_input, np.ndarray):
            arr = image_input
            if arr.dtype == object:
                raise ValueError("Array has dtype=object. Expected numeric array.")
            if arr.dtype != np.uint8:
                arr = (arr * 255).astype(np.uint8) if arr.max() <= 1.0 else arr.astype(np.uint8)
            
            if arr.ndim == 2:
                pil_img = Image.fromarray(arr).convert("RGB")
            elif arr.ndim == 3:
                if arr.shape[2] == 1:
                    pil_img = Image.fromarray(arr.squeeze(axis=2)).convert("RGB")
                elif arr.shape[2] == 3:
                    pil_img = Image.fromarray(arr).convert("RGB")
                elif arr.shape[0] == 3:
                    pil_img = Image.fromarray(arr.transpose(1, 2, 0)).convert("RGB")
                else:
                    raise ValueError(f"Unsupported array shape: {arr.shape}")
            else:
                raise ValueError(f"Unsupported array dimensions: {arr.ndim}")
                
            temp_fd, temp_path = tempfile.mkstemp(suffix=".jpg")
            os.close(temp_fd)
            pil_img.save(temp_path, "JPEG", quality=95)
            image_path = temp_path
            temp_file = temp_path
        elif isinstance(image_input, Image.Image):
            temp_fd, temp_path = tempfile.mkstemp(suffix=".jpg")
            os.close(temp_fd)
            image_input.convert("RGB").save(temp_path, "JPEG", quality=95)
            image_path = temp_path
            temp_file = temp_path
        else:
            raise TypeError("Input must be a file path (str), NumPy array, or PIL Image.")

        if previous_output is None:
            prompt = (
                "Analyze this low-light image for enhancement. "
                "Suggest gamma (0.7-1.2), Alpha-s (0.7-1.2) and Alpha-i (0.7-1.2) parameters. "
                "Respond strictly in this format:\n"
                "Suggested: gamma=X.XX, alpha_s=X.XX, alpha_i=X.XX"
            )
        else:
            prompt = (
                f"Analyze this low-light image for enhancement. Consider previous context: {previous_output}. "
                "Suggest updated enhancement parameters to improve performance. "
                "Respond strictly in this format:\n"
                "Suggested: gamma=X.XX, alpha_s=X.XX, alpha_i=X.XX"
            )

        response = ollama.chat(
            model="gemma3:4b",
            messages=[{"role": "user", "content": prompt, "images": [image_path]}]
        )
        return response["message"]["content"]

    except Exception as e:
        error_msg = f"Error in query_gpt_llava: {str(e)}"
        print(error_msg)
        return error_msg
    finally:
        if temp_file and os.path.exists(temp_file):
            try:
                os.remove(temp_file)
            except Exception:
                pass

# 3. Library Imports & Metrics

In [6]:
import numpy as np
import json
import re
from langgraph.graph import StateGraph
from langchain_core.prompts import PromptTemplate
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image as PILImage
from tqdm import tqdm
import pyiqa

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

brisque_metric = pyiqa.create_metric('brisque', device=device)
niqe_metric = pyiqa.create_metric('niqe', device=device)

def reference_free_metric(img):
    if isinstance(img, str):
        img = PILImage.open(img).convert('RGB')
    if isinstance(img, np.ndarray):
        img = PILImage.fromarray(img.astype(np.uint8)).convert('RGB')
    
    tensor = TF.to_tensor(img).unsqueeze(0).to(device)

    with torch.no_grad():
        brisque_score = brisque_metric(tensor).item()
        niqe_score = niqe_metric(tensor).item()
    return brisque_score, niqe_score

Using device: cuda


# 4. CIDNet Model Definition

In [8]:
pi = 3.141592653589793

class RGB_HVI(nn.Module):
    def __init__(self):
        super(RGB_HVI, self).__init__()
        self.density_k = torch.nn.Parameter(torch.full([1], 0.2))
        self.gated = False
        self.gated2 = False
        self.alpha = 1.0
        self.alpha_s = 1.3
        self.this_k = 0

    def HVIT(self, img):
        eps = 1e-8
        device = img.device
        dtypes = img.dtype
        hue = torch.Tensor(img.shape[0], img.shape[2], img.shape[3]).to(device).to(dtypes)
        value = img.max(1)[0].to(dtypes)
        img_min = img.min(1)[0].to(dtypes)
        hue[img[:, 2] == value] = 4.0 + ((img[:, 0] - img[:, 1]) / (value - img_min + eps))[img[:, 2] == value]
        hue[img[:, 1] == value] = 2.0 + ((img[:, 2] - img[:, 0]) / (value - img_min + eps))[img[:, 1] == value]
        hue[img[:, 0] == value] = (0.0 + ((img[:, 1] - img[:, 2]) / (value - img_min + eps))[img[:, 0] == value]) % 6
        hue[img.min(1)[0] == value] = 0.0
        hue = hue / 6.0
        saturation = (value - img_min) / (value + eps)
        saturation[value == 0] = 0
        hue = hue.unsqueeze(1)
        saturation = saturation.unsqueeze(1)
        value = value.unsqueeze(1)
        k = self.density_k
        self.this_k = k.item()
        color_sensitive = ((value * 0.5 * pi).sin() + eps).pow(k)
        ch = (2.0 * pi * hue).cos()
        cv = (2.0 * pi * hue).sin()
        H = color_sensitive * saturation * ch
        V = color_sensitive * saturation * cv
        I = value
        xyz = torch.cat([H, V, I], dim=1)
        return xyz

    def PHVIT(self, img):
        eps = 1e-8
        H, V, I = img[:, 0, :, :], img[:, 1, :, :], img[:, 2, :, :]
        H = torch.clamp(H, -1, 1)
        V = torch.clamp(V, -1, 1)
        I = torch.clamp(I, 0, 1)
        v = I
        k = self.this_k
        color_sensitive = ((v * 0.5 * pi).sin() + eps).pow(k)
        H = H / (color_sensitive + eps)
        V = V / (color_sensitive + eps)
        H = torch.clamp(H, -1, 1)
        V = torch.clamp(V, -1, 1)
        h = torch.atan2(V + eps, H + eps) / (2 * pi)
        h = h % 1
        s = torch.sqrt(H ** 2 + V ** 2 + eps)
        if self.gated:
            s = s * self.alpha_s
        s = torch.clamp(s, 0, 1)
        v = torch.clamp(v, 0, 1)
        r, g, b = torch.zeros_like(h), torch.zeros_like(h), torch.zeros_like(h)
        hi = torch.floor(h * 6.0)
        f = h * 6.0 - hi
        p = v * (1. - s)
        q = v * (1. - (f * s))
        t = v * (1. - ((1. - f) * s))
        hi0, hi1, hi2, hi3, hi4, hi5 = hi == 0, hi == 1, hi == 2, hi == 3, hi == 4, hi == 5
        r[hi0], g[hi0], b[hi0] = v[hi0], t[hi0], p[hi0]
        r[hi1], g[hi1], b[hi1] = q[hi1], v[hi1], p[hi1]
        r[hi2], g[hi2], b[hi2] = p[hi2], v[hi2], t[hi2]
        r[hi3], g[hi3], b[hi3] = p[hi3], q[hi3], v[hi3]
        r[hi4], g[hi4], b[hi4] = t[hi4], p[hi4], v[hi4]
        r[hi5], g[hi5], b[hi5] = v[hi5], p[hi5], q[hi5]
        rgb = torch.cat([r.unsqueeze(1), g.unsqueeze(1), b.unsqueeze(1)], dim=1)
        if self.gated2:
            rgb = rgb * self.alpha
        return rgb

class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_first"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        self.normalized_shape = (normalized_shape,)

    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None] * x + self.bias[:, None, None]
            return x

class NormDownsample(nn.Module):
    def __init__(self, in_ch, out_ch, scale=0.5, use_norm=False):
        super(NormDownsample, self).__init__()
        self.use_norm = use_norm
        if self.use_norm:
            self.norm = LayerNorm(out_ch)
        self.prelu = nn.PReLU()
        self.down = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.UpsamplingBilinear2d(scale_factor=scale))

    def forward(self, x):
        x = self.down(x)
        x = self.prelu(x)
        if self.use_norm:
            x = self.norm(x)
        return x

class NormUpsample(nn.Module):
    def __init__(self, in_ch, out_ch, scale=2, use_norm=False):
        super(NormUpsample, self).__init__()
        self.use_norm = use_norm
        if self.use_norm:
            self.norm = LayerNorm(out_ch)
        self.prelu = nn.PReLU()
        self.up_scale = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.UpsamplingBilinear2d(scale_factor=scale))
        self.up = nn.Conv2d(out_ch * 2, out_ch, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x, y):
        x = self.up_scale(x)
        x = torch.cat([x, y], dim=1)
        x = self.up(x)
        x = self.prelu(x)
        if self.use_norm:
            return self.norm(x)
        return x

from einops import rearrange

class CAB(nn.Module):
    def __init__(self, dim, num_heads, bias):
        super(CAB, self).__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.q = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)
        self.q_dwconv = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, groups=dim, bias=bias)
        self.kv = nn.Conv2d(dim, dim * 2, kernel_size=1, bias=bias)
        self.kv_dwconv = nn.Conv2d(dim * 2, dim * 2, kernel_size=3, stride=1, padding=1, groups=dim * 2, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)

    def forward(self, x, y):
        b, c, h, w = x.shape
        q = self.q_dwconv(self.q(x))
        kv = self.kv_dwconv(self.kv(y))
        k, v = kv.chunk(2, dim=1)
        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        q = torch.nn.functional.normalize(q, dim=-1)
        k = torch.nn.functional.normalize(k, dim=-1)
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = nn.functional.softmax(attn, dim=-1)
        out = (attn @ v)
        out = rearrange(out, 'b head c (h w) -> b (head c) h w', head=self.num_heads, h=h, w=w)
        out = self.project_out(out)
        return out

class IEL(nn.Module):
    def __init__(self, dim, ffn_expansion_factor=2.66, bias=False):
        super(IEL, self).__init__()
        hidden_features = int(dim * ffn_expansion_factor)
        self.project_in = nn.Conv2d(dim, hidden_features * 2, kernel_size=1, bias=bias)
        self.dwconv = nn.Conv2d(hidden_features * 2, hidden_features * 2, kernel_size=3, stride=1, padding=1, groups=hidden_features * 2, bias=bias)
        self.dwconv1 = nn.Conv2d(hidden_features, hidden_features, kernel_size=3, stride=1, padding=1, groups=hidden_features, bias=bias)
        self.dwconv2 = nn.Conv2d(hidden_features, hidden_features, kernel_size=3, stride=1, padding=1, groups=hidden_features, bias=bias)
        self.project_out = nn.Conv2d(hidden_features, dim, kernel_size=1, bias=bias)
        self.Tanh = nn.Tanh()

    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        x1 = self.Tanh(self.dwconv1(x1)) + x1
        x2 = self.Tanh(self.dwconv2(x2)) + x2
        x = x1 * x2
        x = self.project_out(x)
        return x

class HV_LCA(nn.Module):
    def __init__(self, dim, num_heads, bias=False):
        super(HV_LCA, self).__init__()
        self.gdfn = IEL(dim)
        self.norm = LayerNorm(dim)
        self.ffn = CAB(dim, num_heads, bias)

    def forward(self, x, y):
        x = x + self.ffn(self.norm(x), self.norm(y))
        x = self.gdfn(self.norm(x))
        return x

class I_LCA(nn.Module):
    def __init__(self, dim, num_heads, bias=False):
        super(I_LCA, self).__init__()
        self.norm = LayerNorm(dim)
        self.gdfn = IEL(dim)
        self.ffn = CAB(dim, num_heads, bias=bias)

    def forward(self, x, y):
        x = x + self.ffn(self.norm(x), self.norm(y))
        x = x + self.gdfn(self.norm(x))
        return x

class CIDNet(nn.Module):
    def __init__(self, channels=[36, 36, 72, 144], heads=[1, 2, 4, 8], norm=False):
        super(CIDNet, self).__init__()
        ch1, ch2, ch3, ch4 = channels
        head1, head2, head3, head4 = heads

        self.HVE_block0 = nn.Sequential(nn.ReplicationPad2d(1), nn.Conv2d(3, ch1, 3, stride=1, padding=0, bias=False))
        self.HVE_block1 = NormDownsample(ch1, ch2, use_norm=norm)
        self.HVE_block2 = NormDownsample(ch2, ch3, use_norm=norm)
        self.HVE_block3 = NormDownsample(ch3, ch4, use_norm=norm)

        self.HVD_block3 = NormUpsample(ch4, ch3, use_norm=norm)
        self.HVD_block2 = NormUpsample(ch3, ch2, use_norm=norm)
        self.HVD_block1 = NormUpsample(ch2, ch1, use_norm=norm)
        self.HVD_block0 = nn.Sequential(nn.ReplicationPad2d(1), nn.Conv2d(ch1, 2, 3, stride=1, padding=0, bias=False))

        self.IE_block0 = nn.Sequential(nn.ReplicationPad2d(1), nn.Conv2d(1, ch1, 3, stride=1, padding=0, bias=False))
        self.IE_block1 = NormDownsample(ch1, ch2, use_norm=norm)
        self.IE_block2 = NormDownsample(ch2, ch3, use_norm=norm)
        self.IE_block3 = NormDownsample(ch3, ch4, use_norm=norm)

        self.ID_block3 = NormUpsample(ch4, ch3, use_norm=norm)
        self.ID_block2 = NormUpsample(ch3, ch2, use_norm=norm)
        self.ID_block1 = NormUpsample(ch2, ch1, use_norm=norm)
        self.ID_block0 = nn.Sequential(nn.ReplicationPad2d(1), nn.Conv2d(ch1, 1, 3, stride=1, padding=0, bias=False))

        self.HV_LCA1 = HV_LCA(ch2, head2)
        self.HV_LCA2 = HV_LCA(ch3, head3)
        self.HV_LCA3 = HV_LCA(ch4, head4)
        self.HV_LCA4 = HV_LCA(ch4, head4)
        self.HV_LCA5 = HV_LCA(ch3, head3)
        self.HV_LCA6 = HV_LCA(ch2, head2)

        self.I_LCA1 = I_LCA(ch2, head2)
        self.I_LCA2 = I_LCA(ch3, head3)
        self.I_LCA3 = I_LCA(ch4, head4)
        self.I_LCA4 = I_LCA(ch4, head4)
        self.I_LCA5 = I_LCA(ch3, head3)
        self.I_LCA6 = I_LCA(ch2, head2)

        self.trans = RGB_HVI()

    def forward(self, x):
        dtypes = x.dtype
        hvi = self.trans.HVIT(x)
        i = hvi[:, 2, :, :].unsqueeze(1).to(dtypes)

        i_enc0 = self.IE_block0(i)
        i_enc1 = self.IE_block1(i_enc0)
        hv_0 = self.HVE_block0(hvi)
        hv_1 = self.HVE_block1(hv_0)
        i_jump0, hv_jump0 = i_enc0, hv_0

        i_enc2 = self.I_LCA1(i_enc1, hv_1)
        hv_2 = self.HV_LCA1(hv_1, i_enc1)
        v_jump1, hv_jump1 = i_enc2, hv_2
        i_enc2 = self.IE_block2(i_enc2)
        hv_2 = self.HVE_block2(hv_2)

        i_enc3 = self.I_LCA2(i_enc2, hv_2)
        hv_3 = self.HV_LCA2(hv_2, i_enc2)
        v_jump2, hv_jump2 = i_enc3, hv_3
        i_enc3 = self.IE_block3(i_enc2)
        hv_3 = self.HVE_block3(hv_2)

        i_enc4 = self.I_LCA3(i_enc3, hv_3)
        hv_4 = self.HV_LCA3(hv_3, i_enc3)

        i_dec4 = self.I_LCA4(i_enc4, hv_4)
        hv_4 = self.HV_LCA4(hv_4, i_enc4)

        hv_3 = self.HVD_block3(hv_4, hv_jump2)
        i_dec3 = self.ID_block3(i_dec4, v_jump2)
        i_dec2 = self.I_LCA5(i_dec3, hv_3)
        hv_2 = self.HV_LCA5(hv_3, i_dec3)

        hv_2 = self.HVD_block2(hv_2, hv_jump1)
        i_dec2 = self.ID_block2(i_dec3, v_jump1)

        i_dec1 = self.I_LCA6(i_dec2, hv_2)
        hv_1 = self.HV_LCA6(hv_2, i_dec2)

        i_dec1 = self.ID_block1(i_dec1, i_jump0)
        i_dec0 = self.ID_block0(i_dec1)
        hv_1 = self.HVD_block1(hv_1, hv_jump0)
        hv_0 = self.HVD_block0(hv_1)

        output_hvi = torch.cat([hv_0, i_dec0], dim=1) + hvi
        output_rgb = self.trans.PHVIT(output_hvi)
        return output_rgb

# 5. CIDNet Initialization & Filtering Function (OPTIMIZED)

In [9]:
import torchvision.transforms as transforms

CIDNET_path = "C:/Users/user/Desktop/signalimageprocessing/lolcidnetwieghtbin/lolv1/wprec.bin"

eval_net = CIDNet().cuda()
eval_net.trans.gated = True
eval_net.trans.gated2 = True

eval_net.load_state_dict(torch.load(CIDNET_path, map_location='cuda'))
eval_net.eval()

def filtering(input_img, gamma, alpha_s=1.0, alpha_i=1.0):
    torch.set_grad_enabled(False)
    
    if isinstance(input_img, str):
        input_img = PILImage.open(input_img).convert('RGB')
    elif isinstance(input_img, np.ndarray):
        input_img = PILImage.fromarray(input_img).convert('RGB')
    elif not isinstance(input_img, PILImage.Image):
        raise TypeError("input_img must be a PIL Image, numpy array, or string path")

    pil2tensor = transforms.Compose([transforms.ToTensor()])
    input_tensor = pil2tensor(input_img)
    
    factor = 8
    h, w = input_tensor.shape[1], input_tensor.shape[2]
    H, W = ((h + factor) // factor) * factor, ((w + factor) // factor) * factor
    padh = H - h if h % factor != 0 else 0
    padw = W - w if w % factor != 0 else 0
    input_tensor = F.pad(input_tensor.unsqueeze(0), (0, padw, 0, padh), 'reflect').cuda()

    with torch.no_grad():
        eval_net.trans.alpha_s = alpha_s
        eval_net.trans.alpha = alpha_i
        output = eval_net(input_tensor ** gamma)

    output = torch.clamp(output, 0, 1)
    output = output[:, :, :h, :w]
    enhanced_img = transforms.ToPILImage()(output.squeeze(0).cpu())
    return np.array(enhanced_img.convert('RGB'))

# 6. Prompts (Reference-Free Only)

In [10]:
prompt_template = PromptTemplate(
    input_variables=["imagereport", "brisque_current", "niqe_current"],
    template="""<start_of_turn>user
You are tuning parameters to enhance a low-light image using a predefined function `filtering(image, gamma, alpha_s, alpha_i)` aimed at improving visual quality.

### Step 1: Analyze Current Quality Metrics (Reference-Free)
- BRISQUE: {brisque_current}
- NIQE: {niqe_current}
(Lower values indicate better perceptual quality.)

### Step 2: Visual Analysis (from Vision Assistant)
{imagereport}

### Step 3: Parameter Ranges
- gamma: [0.7, 1.2] — Lower is lighter
- alpha_s: [0.7, 1.2] — Higher is more saturated
- alpha_i: [0.7, 1.2] — Higher is lighter

### Step 4: Your Task
Intelligently select values for gamma, alpha_s, and alpha_i that would likely **decrease** both BRISQUE and NIQE. 

def enhancing_image(image_path):
    enhanced_img = filtering(image_path, gamma=?, alpha_s=?, alpha_i=?)
    return enhanced_img

Only return the fully defined function. Do not include comments or explanations.<end_of_turn><start_of_turn>model""")

In [11]:
feedback_prompt_template = PromptTemplate(
    input_variables=["generated_code", "error_generated_code", "brisque_generated", "niqe_generated", 
                     "imagereport_generate", "trend", "patience_counter", "patience_maximum", "final_code"],
    template="""<start_of_turn>user
You are evaluating the performance of an image enhancement function.

### Step 1: Check Current Output Quality
- BRISQUE: {brisque_generated}
- NIQE: {niqe_generated}
(Lower is better.)
- Visual Analysis: {imagereport_generate}
- Performance trend: {trend}

### Step 2: Inspect the Code and Errors
{generated_code}
Errors (if any): {error_generated_code}
Best code so far: {final_code}

### Step 3: Parameter Ranges
- gamma: [0.7,1.2] — Lower is lighter
- Alpha-s: [0.7,1.2] — Higher is more saturated
- Alpha-i: [0.7,1.2] — Higher is lighter

### Final Task:
Update the function below with specific gamma, alpha_s, and alpha_i values to improve the scores.

def enhancing_image(image_path):
    enhanced_img = filtering(image_path, gamma=?, alpha_s=?, alpha_i=?)
    return enhanced_img

Only provide the final function. Do not include any reasoning.<end_of_turn><start_of_turn>model""")

In [12]:
refinement_prompt_template = PromptTemplate(
    input_variables=["generated_code", "feedback", "error_generated_code", "brisque_generated", 
                     "niqe_generated", "imagereport_generate", "trend", "patience_counter", "patience_maximum"],
    template="""<start_of_turn>user
Refine the image enhancement function to **minimize** BRISQUE and NIQE.

### Step 1: Performance Summary
- Achieved BRISQUE: {brisque_generated}
- Achieved NIQE: {niqe_generated}
- Performance trend: {trend}

### Step 2: Visual Analysis and Feedback
- Visual Analysis: {imagereport_generate}
- Feedback: {feedback}

### Step 3: Decide on Next Action
- If trend = "improved", keep refining in the same direction.
- If trend = "worsened", adjust parameters in the opposite direction.
- If trend = "no_change", try a new parameter combination.

### Step 4: Parameter Ranges
- gamma: [0.7,1.2] — Lower is lighter
- Alpha-s: [0.7,1.2] — Higher is more saturated
- Alpha-i: [0.7,1.2] — Higher is lighter

### Final Output:
def enhancing_image(image_path):
    enhanced_img = filtering(image_path, gamma=?, alpha_s=?, alpha_i=?)
    return enhanced_img

Return only the updated function.<end_of_turn><start_of_turn>model""")

# 7. Helper Functions & State Management

In [13]:
def parse_gamma_alpha_scores_as_string(response):
    clean_response = response.strip()
    if '[/INST]' in clean_response:
        clean_response = clean_response.split('[/INST]')[-1].strip()
    float_pattern = r'([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)'
    patterns = {
        'gamma': fr'gamma[=\s:]*{float_pattern}',
        'alpha_s': fr'alpha_s[=\s:]*{float_pattern}',
        'alpha_i': fr'alpha_i[=\s:]*{float_pattern}',
    }
    result = {'gamma': None, 'alpha_s': None, 'alpha_i': None}
    for key, pattern in patterns.items():
        match = re.search(pattern, clean_response, re.IGNORECASE)
        if match:
            try:
                result[key] = float(match.group(1))
            except ValueError:
                pass
    return f"gamma={result['gamma']}, alpha_s={result['alpha_s']}, alpha_i={result['alpha_i']}"

In [14]:
class LongTermMemory:
    def __init__(self):
        self.memory = {}
    def set(self, image_name, best_score):
        self.memory[image_name] = best_score
    def get(self, image_name):
        return self.memory.get(image_name, None)
    def get_all(self):
        return self.memory

ltm = LongTermMemory()

In [15]:
def current_example_metric(image_path, filename, dataset_name=""):
    imagereport = query_gpt_llava(image_path)
    print(f"Initial Report: {imagereport}")
    brisque_low, niqe_low = reference_free_metric(image_path)

    return {
        "image_path": image_path,
        "filename": filename,
        "dataset_name": dataset_name,
        "brisque_current": brisque_low,
        "niqe_current": niqe_low,
        "imagereport": imagereport,
        "previous_best": ltm.get_all()
    }

In [16]:
def code_check_updated(llm_code, img):
    processed_code = ""
    status_message = None
    try:
        # 🛠️ ROBUST REGEX FIX:
        # 1. Try to extract from markdown code blocks first. Safely captures docstrings.
        code_blocks = re.findall(r"```(?:python)?\s*(.*?)```", llm_code, re.DOTALL | re.IGNORECASE)
        if code_blocks:
            processed_code = code_blocks[0].strip()
        else:
            # 2. Fallback: If no markdown blocks exist, capture from 'def enhancing_image' to the end.
            match = re.search(r"(def\s+enhancing_image\b.*)", llm_code, re.DOTALL)
            if match:
                processed_code = match.group(1).strip()
            else:
                return "", "No valid enhancing_image function found in LLM output"

        namespace = {'np': np, 'filtering': filtering}
        exec(processed_code, namespace)

        if "enhancing_image" not in namespace:
            status_message = "No enhancing_image function defined"
        else:
            try:
                result = namespace["enhancing_image"](img)
                if isinstance(result, (np.ndarray, PILImage.Image)):
                    status_message = "Code run successfully without error"
                else:
                    status_message = "enhancing_image did not return a valid image"
            except Exception as e:
                status_message = f"Error running enhancing_image function: {str(e)}"
    except Exception as e:
        status_message = f"Error executing LLM-generated code: {str(e)}"
        
    return processed_code, status_message

def apply_llm_generated_code_update(img, llm_code):
    processed_code, status = code_check_updated(llm_code, img)
    if status != "Code run successfully without error":
        print("LLM code status:", status)
        if isinstance(img, str):
            return np.array(PILImage.open(img).convert('RGB'))
        elif isinstance(img, PILImage.Image):
            return np.array(img.convert('RGB'))
        return img if isinstance(img, np.ndarray) else img

    local_vars = {'np': np, 'filtering': filtering}
    try:
        exec(processed_code, local_vars)
        result = local_vars["enhancing_image"](img)
        if isinstance(result, PILImage.Image):
            return np.array(result.convert('RGB'))
        elif isinstance(result, np.ndarray):
            return result
    except Exception as e:
        print(f"Runtime error in enhancing_image: {e}")
        
    if isinstance(img, str):
        return np.array(PILImage.open(img).convert('RGB'))
    elif isinstance(img, PILImage.Image):
        return np.array(img.convert('RGB'))
    return img

# 8. Adaptive Patience Controller

In [17]:
class SimplePatienceController:
    def __init__(self, base_patience=5, max_patience=15, improvement_threshold=1e-3):
        self.base_patience = base_patience
        self.max_patience = max_patience
        self.improvement_threshold = improvement_threshold
        self.reset()

    def reset(self):
        self.best_metrics = None
        self.best_code = None
        self.iterations_without_improvement = 0
        self.current_patience = self.base_patience
        self.performance_history = []
        self.initial_metrics = None
        print(f"  [Patience] 🔄 Reset - patience: {self.current_patience}")

    def decide_continuation(self, current_metrics, current_code, iteration):
        self.performance_history.append(current_metrics.copy())
        
        if self.best_metrics is None:
            self.best_metrics = current_metrics.copy()
            self.best_code = current_code
            self.initial_metrics = current_metrics.copy()
            print(f"  [Patience] Initial - BRISQUE: {current_metrics['brisque']:.4f}, NIQE: {current_metrics['niqe']:.4f}")
            return True

        improved = False
        details = []
        
        if current_metrics['brisque'] < self.best_metrics['brisque'] - self.improvement_threshold:
            improved = True
            details.append(f"BRISQUE: {self.best_metrics['brisque']:.4f} → {current_metrics['brisque']:.4f}")
            self.best_metrics['brisque'] = current_metrics['brisque']
            
        if current_metrics['niqe'] < self.best_metrics['niqe'] - self.improvement_threshold:
            improved = True
            details.append(f"NIQE: {self.best_metrics['niqe']:.4f} → {current_metrics['niqe']:.4f}")
            self.best_metrics['niqe'] = current_metrics['niqe']

        if improved:
            self.best_code = current_code
            self.iterations_without_improvement = 0
            self.current_patience = min(self.max_patience, self.current_patience + 1)
            print(f"  [Patience] IMPROVED! {', '.join(details)}")
        else:
            self.iterations_without_improvement += 1
            print(f"  [Patience] ⏸ No improvement ({self.iterations_without_improvement}/{self.current_patience})")

        if self.iterations_without_improvement >= self.current_patience or iteration >= 90:
            print(f"  [Patience] STOPPING after {iteration+1} iterations")
            return False
        return True

    def get_summary(self):
        if not self.performance_history or not self.initial_metrics or not self.best_metrics:
            return "No data"
        return {
            'iterations': len(self.performance_history),
            'initial_brisque': self.initial_metrics['brisque'],
            'initial_niqe': self.initial_metrics['niqe'],
            'best_brisque': self.best_metrics['brisque'],
            'best_niqe': self.best_metrics['niqe'],
            'improvement_brisque': self.initial_metrics['brisque'] - self.best_metrics['brisque'],
            'improvement_niqe': self.initial_metrics['niqe'] - self.best_metrics['niqe'],
            'best_code': self.best_code
        }

patience_controller = SimplePatienceController()

  [Patience] 🔄 Reset - patience: 5


# 9. LangGraph Construction

In [18]:
def initialize_state(inputs):
    inputs["patience_counter"] = 0
    inputs["patience_maximum"] = 1
    inputs["initial_brisque"] = inputs.get("brisque_current", 1e6)
    inputs["initial_niqe"] = inputs.get("niqe_current", 1e6)
    inputs["trend"] = "initial"
    inputs["final_code"] = "None"
    inputs["imagereport_generate"] = "None"
    inputs["lava_trend"] = None
    return inputs

def generate_code(inputs):
    prompt = prompt_template.format(**inputs)
    inputs["generated_code"] = query_gpt_qwen(prompt)
    print(f"Generated code: {inputs['generated_code']}\n")
    return inputs

def feedback_prompt(inputs):
    latest_code = inputs.get("refined_code", inputs["generated_code"])
    understandable_code, inputs["error_generated_code"] = code_check_updated(latest_code, inputs["image_path"])

    try:
        if inputs["error_generated_code"] == "Code run successfully without error":
            generated_image = apply_llm_generated_code_update(inputs["image_path"], understandable_code)
            brisque_gen, niqe_gen = reference_free_metric(generated_image)
            imagereport_generate = query_gpt_llava(generated_image, inputs["lava_trend"])
            inputs['refine_img_inside'] = generated_image
            inputs['original_img_inside'] = PILImage.open(inputs["image_path"]).convert('RGB')
        else:
            imagereport_generate = "Code Error"
            brisque_gen = niqe_gen = 1e6
    except Exception as e:
        imagereport_generate = f"Error: {str(e)}"
        brisque_gen = niqe_gen = 1e6

    prompt = feedback_prompt_template.format(
        generated_code=understandable_code,
        error_generated_code=inputs["error_generated_code"],
        brisque_generated=brisque_gen,
        niqe_generated=niqe_gen,
        imagereport_generate=imagereport_generate,
        trend=inputs["trend"],
        patience_counter=inputs["patience_counter"],
        patience_maximum=inputs["patience_maximum"],
        final_code=inputs["final_code"]
    )
    inputs["feedback"] = query_gpt_qwen(prompt)
    inputs["brisque_generated"] = brisque_gen
    inputs["niqe_generated"] = niqe_gen
    inputs['imagereport_generate'] = imagereport_generate
    return inputs

def refine_prompt(inputs):
    prompt = refinement_prompt_template.format(
        generated_code=inputs.get("refined_code", inputs["generated_code"]),
        feedback=inputs["feedback"],
        error_generated_code=inputs["error_generated_code"],
        brisque_generated=inputs["brisque_generated"],
        niqe_generated=inputs["niqe_generated"],
        imagereport_generate=inputs['imagereport_generate'],
        trend=inputs["trend"],
        patience_counter=inputs["patience_counter"],
        patience_maximum=inputs["patience_maximum"]
    )
    inputs["refined_code"] = query_gpt_qwen(prompt)

    understandable_code, inputs["error_generated_code"] = code_check_updated(inputs["refined_code"], inputs["image_path"])
    try:
        if inputs["error_generated_code"] == "Code run successfully without error":
            refined_image = apply_llm_generated_code_update(inputs["image_path"], understandable_code)
            brisque_ref, niqe_ref = reference_free_metric(refined_image)
            refinedimagereport_generate = query_gpt_llava(refined_image, inputs["lava_trend"])
            inputs['refine_img_inside'] = refined_image
        else:
            refinedimagereport_generate = "Code Error"
            brisque_ref = niqe_ref = 1e6
    except Exception as e:
        refinedimagereport_generate = f"Error: {str(e)}"
        brisque_ref = niqe_ref = 1e6

    inputs["brisque_refined"] = brisque_ref
    inputs["niqe_refined"] = niqe_ref
    inputs['refinedimagereport_generate'] = refinedimagereport_generate

    TOL = 1e-3
    improvements = worsenings = 0
    if inputs["brisque_refined"] < inputs["initial_brisque"] - TOL: improvements += 1
    elif inputs["brisque_refined"] > inputs["initial_brisque"] + TOL: worsenings += 1
    if inputs["niqe_refined"] < inputs["initial_niqe"] - TOL: improvements += 1
    elif inputs["niqe_refined"] > inputs["initial_niqe"] + TOL: worsenings += 1

    if improvements > 0 and worsenings == 0:
        inputs["initial_brisque"] = inputs["brisque_refined"]
        inputs["initial_niqe"] = inputs["niqe_refined"]
        inputs["final_code"] = inputs["refined_code"]
        inputs["trend"] = "improved"
    elif worsenings > 0 and improvements == 0:
        inputs["patience_counter"] += 1
        inputs["trend"] = "worsened"
    elif improvements == 0 and worsenings == 0:
        inputs["trend"] = "no_change"
    else:
        inputs["trend"] = "mixed"

    inputs["lava_trend"] = (
        f"Previously, performance was {inputs['trend']} when {parse_gamma_alpha_scores_as_string(inputs['refined_code'])} "
        f"gave BRISQUE: {inputs['brisque_refined']:.4f}, NIQE: {inputs['niqe_refined']:.4f}. "
        f"Best so far: {parse_gamma_alpha_scores_as_string(inputs['final_code'])} "
        f"gave BRISQUE: {inputs['initial_brisque']:.4f}, NIQE: {inputs['initial_niqe']:.4f}."
    )
    print(f"Trend: {inputs['trend']} | Patience: {inputs['patience_counter']} | BRISQUE: {inputs['brisque_refined']:.4f} | NIQE: {inputs['niqe_refined']:.4f}")
    return inputs

def should_continue(inputs):
    """Pure routing function: evaluates metrics and decides the next node."""
    current_metrics = {
        'brisque': float(inputs.get("brisque_refined", 1e6)),
        'niqe': float(inputs.get("niqe_refined", 1e6))
    }
    iteration = inputs.get("patience_counter", 0)
    
    current_code = (inputs.get("refined_code") or 
                    inputs.get("generated_code") or 
                    "def enhancing_image(image_path):\n    return filtering(image_path, gamma=1.0, alpha_s=1.0, alpha_i=1.0)")
    
    if patience_controller.decide_continuation(current_metrics, current_code, iteration):
        return "feedback_prompt"
    else:
        summary = patience_controller.get_summary()
        if isinstance(summary, dict):
            print(f"\n  Image Summary: Initial B:{summary['initial_brisque']:.4f}/N:{summary['initial_niqe']:.4f} -> Best B:{summary['best_brisque']:.4f}/N:{summary['best_niqe']:.4f}")
        
        return "end"

def end_node(inputs):
    """Final node: safely retrieves the best code and updates the state dictionary."""
    summary = patience_controller.get_summary()
    
    if isinstance(summary, dict):
        best_code = summary.get('best_code')
        
        if not best_code or str(best_code).strip().lower() == "none":
            print("  [Safety Fallback] best_code was invalid, using generated_code instead.")
            best_code = inputs.get("generated_code")
            
        if not best_code or str(best_code).strip().lower() == "none":
            best_code = "def enhancing_image(image_path):\n    return filtering(image_path, gamma=1.0, alpha_s=1.0, alpha_i=1.0)"
            
        inputs["final_code"] = best_code
        print(f"Process complete. Final code:\n{inputs['final_code']}")
    else:
        print("Process complete. Final code: None (No summary available)")
        inputs["final_code"] = "def enhancing_image(image_path):\n    return filtering(image_path, gamma=1.0, alpha_s=1.0, alpha_i=1.0)"

    inputs["previous_best"] = parse_gamma_alpha_scores_as_string(inputs["final_code"])
    return inputs

graph = StateGraph(dict)
graph.add_node("initialize_state", initialize_state)
graph.add_node("generate_code", generate_code)
graph.add_node("feedback_prompt", feedback_prompt)
graph.add_node("refine_prompt", refine_prompt)
graph.add_node("end", end_node)

graph.add_edge("initialize_state", "generate_code")
graph.add_edge("generate_code", "feedback_prompt")
graph.add_edge("feedback_prompt", "refine_prompt")
graph.add_conditional_edges("refine_prompt", should_continue, {"end": "end", "feedback_prompt": "feedback_prompt"})
graph.set_entry_point("initialize_state")
graph.set_finish_point("end")
chain = graph.compile()

# 10. Evaluation Loop (Reference-Free Datasets) - OPTIMIZED + IMAGE SAVING

In [19]:
import gc
import nest_asyncio
nest_asyncio.apply()

dataset_base_path = 'C:/Users/user/Desktop/signalimageprocessing/NEW JOURNAL UPDATE/reference-free dataset/dataset/' 
datasets = ['DICM']

results = {ds: {'brisque': [], 'niqe': [], 'time': [], 'count': 0} for ds in datasets}
image_count = 0

for dataset in datasets:
    dataset_dir = os.path.join(dataset_base_path, dataset)
    if not os.path.exists(dataset_dir):
        print(f"⚠️ Dataset directory not found, skipping: {dataset_dir}")
        continue
        
    for dirname, _, filenames in os.walk(dataset_dir):
        for filename in filenames:
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                continue
                
            try:
                patience_controller.reset()
                print(f"\n--- Dataset: {dataset} | Image {image_count}: {filename} ---")
                inference_start_time = time.time()

                img_path = os.path.join(dirname, filename)
                metric_ = current_example_metric(img_path, filename, dataset)
                
                # Run the LangGraph search pipeline
                inputs = chain.invoke(metric_, {"recursion_limit": 1000})

                # 🛠️ OPTIMIZATION: Grab the best scores directly from the controller's memory
                summary = patience_controller.get_summary()
                
                if isinstance(summary, dict):
                    final_brisque = summary['best_brisque']
                    final_niqe = summary['best_niqe']
                    final_code = summary['best_code']
                else:
                    # Ultimate fallback if the graph failed completely
                    final_brisque = metric_['brisque_current']
                    final_niqe = metric_['niqe_current']
                    final_code = "def enhancing_image(image_path):\n    return filtering(image_path, gamma=1.0, alpha_s=1.0, alpha_i=1.0)"

                print(f"\n✅ Final Selected Code:\n{final_code}")
                print(f"✅ Using tracked best scores -> BRISQUE: {final_brisque:.4f}, NIQE: {final_niqe:.4f}\n")

                # 🖼️ NEW: Generate and save the final enhanced image using the best code
                try:
                    enhanced_img_array = apply_llm_generated_code_update(img_path, final_code)
                    
                    # Create output directory next to the original dataset (e.g., .../DICM_enhanced)
                    output_dir = os.path.join(os.path.dirname(dataset_base_path.rstrip('/\\')), f"{dataset}_enhanced")
                    os.makedirs(output_dir, exist_ok=True)
                    
                    # Save the image
                    save_path = os.path.join(output_dir, filename)
                    PILImage.fromarray(enhanced_img_array.astype(np.uint8)).convert('RGB').save(save_path)
                    print(f"💾 Saved enhanced image to: {save_path}")
                except Exception as save_error:
                    print(f"⚠️ Warning: Could not save enhanced image for {filename}: {save_error}")

                # Save the results
                results[dataset]['brisque'].append(final_brisque)
                results[dataset]['niqe'].append(final_niqe)

                inference_end_time = time.time()
                results[dataset]['time'].append(inference_end_time - inference_start_time)
                results[dataset]['count'] += 1

                ltm.set(f"{dataset}_{filename}", f"BRISQUE: {final_brisque:.4f}, NIQE: {final_niqe:.4f}")

                image_count += 1
                torch.cuda.empty_cache()
                gc.collect()

            except Exception as e:
                print(f"Fatal error on image {filename}: {e}")
                continue

  [Patience] 🔄 Reset - patience: 5

--- Dataset: DICM | Image 0: 01.JPG ---
Initial Report: Suggested: gamma=1.1, alpha_s=1.0, alpha_i=0.9
Generated code: ```python
def enhancing_image(image_path):
    enhanced_img = filtering(image_path, gamma=1.15, alpha_s=1.1, alpha_i=1.05)
    return enhanced_img
```

Trend: mixed | Patience: 0 | BRISQUE: 3.7990 | NIQE: 3.8106
  [Patience] Initial - BRISQUE: 3.7990, NIQE: 3.8106
Trend: mixed | Patience: 0 | BRISQUE: 3.2099 | NIQE: 3.8049
  [Patience] IMPROVED! BRISQUE: 3.7990 → 3.2099, NIQE: 3.8106 → 3.8049
Trend: mixed | Patience: 0 | BRISQUE: 5.6686 | NIQE: 3.9520
  [Patience] ⏸ No improvement (1/6)
Trend: mixed | Patience: 0 | BRISQUE: 3.2099 | NIQE: 3.8049
  [Patience] ⏸ No improvement (2/6)
Trend: mixed | Patience: 0 | BRISQUE: 3.2099 | NIQE: 3.8049
  [Patience] ⏸ No improvement (3/6)
Trend: mixed | Patience: 0 | BRISQUE: 5.6686 | NIQE: 3.9520
  [Patience] ⏸ No improvement (4/6)
Trend: mixed | Patience: 0 | BRISQUE: 3.0364 | NIQE: 3.8256
  [Pa

# 11. Final Results Reporting

In [20]:
print("\n" + "="*60)
print("FINAL EVALUATION RESULTS (Reference-Free Metrics)")
print("="*60)

all_brisque, all_niqe, all_time = [], [], []

for dataset, metrics in results.items():
    if len(metrics['brisque']) > 0:
        avg_b = np.mean(metrics['brisque'])
        avg_n = np.mean(metrics['niqe'])
        avg_t = np.mean(metrics['time'])
        
        all_brisque.extend(metrics['brisque'])
        all_niqe.extend(metrics['niqe'])
        all_time.extend(metrics['time'])
        
        print(f"\nDataset: {dataset} ({metrics['count']} images)")
        print(f"  Average BRISQUE: {avg_b:.4f} (lower is better)")
        print(f"  Average NIQE:    {avg_n:.4f} (lower is better)")
        print(f"  Avg Inference Time: {avg_t:.2f}s")
        print("-" * 40)

if len(all_brisque) > 0:
    print("\nOVERALL AVERAGES ACROSS ALL DATASETS:")
    print(f"  Average BRISQUE: {np.mean(all_brisque):.4f}")
    print(f"  Average NIQE:    {np.mean(all_niqe):.4f}")
    print(f"  Average Inference Time: {np.mean(all_time):.2f}s")
    
    np.save('brisque_results.npy', np.array(all_brisque))
    np.save('niqe_results.npy', np.array(all_niqe))
else:
    print("No images were successfully evaluated.")


FINAL EVALUATION RESULTS (Reference-Free Metrics)

Dataset: DICM (69 images)
  Average BRISQUE: 19.4475 (lower is better)
  Average NIQE:    3.4938 (lower is better)
  Avg Inference Time: 142.19s
----------------------------------------

OVERALL AVERAGES ACROSS ALL DATASETS:
  Average BRISQUE: 19.4475
  Average NIQE:    3.4938
  Average Inference Time: 142.19s
